# B1: What do decision features add to a fixed predictor?

Each selected decision model contributes per-text-column, joint or both feature blocks (binary: one probability; multiclass: C probabilities; regression: all nine ordered probabilities; empty text stays missing). Each dataset and predictor uses a no-text baseline and three views per selected decision model. The question this notebook answers first is **how the decision models differ in what they extract**: per-model gains, per-dataset ranks, fold-level win rates and raw Δ per task family; the headroom-normalised gain is one supporting view.

Seven predictors use prespecified settings on the same five outer folds across 18 datasets (3 binary, 6 multiclass, 9 regression). They are linear/logistic regression, CatBoost, LightGBM, TabPFN-3.5, TabICL v2, LimiX-2 and Kumo. Each structured baseline includes the non-text columns and one text-availability flag per text column. Fixed headline metrics: binary ROC-AUC, multiclass macro one-vs-rest ROC-AUC, regression R². Datasets are the unit: each contributes its five-fold mean paired gain; raw gains stay within a task family (ΔAUC and ΔR², N = 9 each, separate panels) and only the headroom share (gain / (1 − baseline)) and the MulTaBench-style normalised score (per dataset and fold, min-max over the compared configurations; negative R² clipped at 0) pool all 18 datasets. Means sit next to medians; intervals resample datasets (2,000 draws). Decision medians are taken over all members of the recorded decision-model cohort; nothing selects a winner.

Reads one compact export, `output_JEVPFN/B/exp04_decision_features/results/group_b/b1_decision_features.<cohort>-<roster digest>.zip` (`scripts/slurm/rq2_group_b_export.slurm B1`); every section shows a one-line waiting state while it is absent.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import pandas as pd
from src.utils.notebook_display import display_json, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.evaluation import benchmark_export, group_b, group_b_views as views, learner_scores, study_export
from src.visualize import critical_difference, figures, style
from src.visualize import group_b as gbfig
style.apply()
NOTEBOOK = "B/exp04_decision_features"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
report_section("1. What is complete: export, provenance and coverage")
display_json("Notebook execution identity", provenance())
frames, receipt = benchmark_export.read("B1")
EXPORTS = {"B1": (frames, receipt)}
ROSTER = benchmark_export.decision_roster()
display_json("Decision-model cohort and members", benchmark_export.cohort_record())
WAITING = "Waiting for export: no measurements are shown."
for study, (frames_, receipt_) in EXPORTS.items():
    print(f"{study}: {study_export.describe(receipt_)}")
    record({k: receipt_.get(k) for k in ("status", "path", "created_utc", "origin", "sha256", "bytes", "schema", "receipt_sha256")}, f"{study} export identity")
    if frames_:
        display_table(f"{study}: planned versus complete five-fold arms per phase and predictor", frames_["coverage"], preview_rows=14)
        if "median_roster_gaps" in frames_ and not frames_["median_roster_gaps"].empty:
            display_table(f"{study}: dataset x predictor cells whose median lacks configured decision models", frames_["median_roster_gaps"])

B1: Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\B\exp04_decision_features\results\group_b\b1_decision_features.open_weight-5baeccd37d70d034.zip is absent. No measurements are shown.


## 2. Summary: the gain landscape

Per dataset, every decision model (grey, both view, median over predictors) and the median model per view (diamonds, with the range over predictors). Classification and regression are separate panels because ΔAUC and ΔR² are not commensurable.

In [2]:
report_section("2. Summary: the gain landscape")
if not frames:
    print(WAITING)
else:
    gains, medians = frames["dataset_gains"], frames["median_gains"]
    model_gains = views.model_gains(gains, roster=ROSTER)
    gbfig.render(save, gbfig.gain_landscape(model_gains, views.median_model_gains(medians), name="b1_gain_landscape", what="Decision features in fixed predictors"))
    display_table("Per dataset, decision model and view: median gain over predictors and its range", model_gains, preview_rows=12)

Waiting for export: no measurements are shown.


## 3. How the decision models differ

Ranks within each dataset (rows ordered by mean rank) with fold-level win rates against no text; the per-predictor heatmap of mean gains per task family; and whether the seven predictors agree on the ranking of the models (Kendall τ_b). Fold-level win rates are descriptive: cells are dataset × predictor × fold, and intervals resample whole datasets.

In [3]:
report_section("3. How the decision models differ")
if not frames:
    print(WAITING)
else:
    from src.evaluation import model_properties
    head = views.fold_long(frames["folds"])
    ranks, rank_summary = views.model_ranks(gains, roster=ROSTER)
    model_wins = views.fold_win_rates(head, [f"decision/both/{p}" for p in ROSTER], by_predictor=False)
    gbfig.render(save, gbfig.model_rank_heatmap(ranks, rank_summary, win_rates=model_wins, profile=model_properties.profile_table(), name="b1_model_ranks", what="Decision models"))
    display_table("Mean within-dataset rank per decision model (dataset-bootstrap 95% interval)", rank_summary, preview_rows=17)
    display_table("Fold-level win rate of each decision model (both) against no text, predictors pooled", model_wins, preview_rows=17)
    gbfig.render(save, gbfig.model_by_predictor(gains, order=list(rank_summary.provider), name="b1_models_by_predictor", what="Decision models by predictor"))
    agreement = views.predictor_rank_agreement(gains, roster=ROSTER)
    gbfig.render(save, gbfig.predictor_rank_agreement(agreement, name="b1_predictor_rank_agreement", what="Ranking agreement between predictors"))
    record(agreement, "Kendall tau_b between predictors")

Waiting for export: no measurements are shown.


## 4. Views, task families and fold-level wins

Per-dataset paired contrasts between views on the 16 datasets with at least two text columns; the equal-dataset mean and median raw gain per task family; the pooled headroom share (all 18 datasets) as a supporting view; the MulTaBench-style normalised score, which puts ROC-AUC and R² on one 0–1 scale (per dataset and fold, min-max over the 28 predictor × arm configurations: no text and the median model per view; negative R² clipped at 0, all-tied folds skipped) next to, never instead of, the per-family panels; and fold-level win rates of the median model per view and predictor.

In [4]:
report_section("4. Views, task families and fold-level wins")
if not frames:
    print(WAITING)
else:
    text_columns = views.text_column_counts()
    contrasts = views.view_contrasts(gains, roster=ROSTER, text_columns=text_columns)
    single = [d for d, n in text_columns.items() if n < 2]
    gbfig.render(save, gbfig.view_contrast(contrasts, name="b1_view_contrasts", what="Decision views", single_text=single))
    modes = ["decision/per_column/median", "decision/joint/median", "decision/both/median"]
    family = views.family_summaries(medians, modes)
    display_table("Median decision model: equal-dataset mean/median gain per task family and pooled headroom share", family, preview_rows=21)
    gbfig.render(save, gbfig.gain_forest(family, modes, name="b1_gain_by_task_family", what="Median decision model per view"))
    gbfig.render(save, gbfig.gain_forest(family, modes, name="b1_headroom_share", what="Median decision model per view", share=True))
    normalized_arms = ["baseline", *modes]
    _, normalized = views.normalized_view(head, normalized_arms, roster=ROSTER)
    gbfig.render(save, gbfig.normalized_score_forest(normalized, normalized_arms, name="b1_normalized_score", what="No text and the median decision model per view"))
    display_table("Normalised score (MulTaBench-style): equal-dataset mean and median per predictor and arm, all datasets and per task family", normalized, preview_rows=28)
    head_median = pd.concat([head, views.median_fold_scores(head, roster=ROSTER)], ignore_index=True)
    view_wins = views.fold_win_rates(head_median, modes)
    gbfig.render(save, gbfig.fold_win_rate_forest(view_wins, modes, name="b1_fold_win_rates", what="Median decision model per view"))
    display_table("Fold-level win rates of the median model per view and predictor", view_wins, preview_rows=24)

Waiting for export: no measurements are shown.


## 5. Statistics

For each predictor, the study compares the decision views with its no-text baseline. Holm adjustment covers that predictor’s comparisons. The ranks, fold win rates and PAMA describe performance across datasets. The primary 21 jointly adjusted results are reported in general/5_synthesis.

In [5]:
report_section("5. Statistics")
framework = learner_scores.run_application("B1_decision_features")
print(f"B1_decision_features: {framework['status']} (framework.status = {framework['framework_status']})")
display_json("Framework application B1_decision_features", learner_scores.summary_record(framework))
gbfig.render(save, critical_difference.cd_figures(framework))
if frames:
    display_table("PAMA over the decision models (both view), dataset x predictor x fold cells", views.model_pama(head, roster=ROSTER), preview_rows=17)

B1_decision_features: awaiting_export (framework.status = approved)


## 6. Regression feature widths and complete report

In [6]:
from src.evaluation.research_questions import regression_width_table
report_section("6. Regression feature widths")
display_table("Nine-probability widths including non-text columns and availability flags", regression_width_table(), preview_rows=9)
print("All nine ordered probabilities are used for regression; no mean-readout arm is planned.")

dataset,dataset_id,text_columns,non_text_columns,joint_probabilities,per_column_probabilities,both_probabilities,kumo_conservative_max
Baby Products,REG_TEXT_BABIES_PRICES,4,8,21,48,57,65
Book Price,REG_TEXT_BOOK_PRICE,5,3,17,53,62,65
Book Readability,REG_TEXT_BOOK_READABILITY,6,24,39,84,93,117
Mercari Marketplace,REG_TEXT_MERCARI_MARKETPLACE,6,3,18,63,72,75
Montgomery Salaries,REG_TEXT_MONTGOMERY_SALARIES,3,8,20,38,47,55
Rotten Tomatoes,REG_TEXT_ROTTEN_TOMATOES,13,2,24,132,141,143
SciMagojr Impact,REG_TEXT_SCIMAGOJR_IMPACT,10,12,31,112,121,133
Vancouver Salaries,REG_TEXT_VANCOUVER_SALARIES,2,3,14,23,32,35
Video Games Sales,REG_TEXT_VIDEO_GAMES_SALES,2,3,14,23,32,35


All nine ordered probabilities are used for regression; no mean-readout arm is planned.


In [7]:
finish_report(NOTEBOOK, save.summary() + "\nOnly the verified exports and local caches named above were read; no inference, model training or VSC changes were performed.")

B/exp04_decision_features

1. What is complete: export, provenance and coverage


Notebook execution identity
{
  "code_sha256": "8b4d9540fb90939d06c1177e768d032c9866801857bd06cfd36cb4fa3aea5040",
  "git_head": "e5d935fe73a631187c6d40435fbeb13c6743c494",
  "git_dirty": true,
  "tfm_library_pin": "4574eea0c2ce6646b5d2e9c7f53a6ef3882c368c dm-library (remotes/origin/HEAD)\n 248f137046487e0d57360abba1e060420ae54e1a tfm-library (remotes/origin/HEAD)\n 2208cf68d6351eff2cb0ddb5a9eb1d822523457b vsc-library (heads/main-8-g2208cf6)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}


Decision-model cohort and members
{
  "decision_cohort": "open_weight",
  "decision_roster": [
    "liquid",
    "liquid_lfm3",
    "perplexity",
    "cloudflare",
    "strands",
    "intern",
    "h2o",
    "sii",
    "kev",
    "decision3",
    "nimble",
    "glide",
    "mammoth",
    "drex_max